# README
- **Author**: RUEI-HSIANG, CHANG (B12303030), Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
整理候選人社群入口。觀察單位是候選人×平台×帳號×來源；姓名＋縣市＋公職種類唯一對應才連接；不依資料缺漏排除候選人。保留多個網址，不擅自解決重複帳號。

## What does this file take?
固定 capture_manifest（逐檔來源 URL／日期／HTTP／hash）、candidate_master_reviewed 名冊與 candidate_campaign_sources 既有來源；各檔 SHA-256 見程式 PINS。Raw 說明：data/raw/social_sources_20261019_documentation.ipynb。

## What does this file output?
- data/processed/candidate_social_account_evidence_20261019.csv：逐來源證據，作為社群入口索引。
- data/processed/meta_candidate_account_worklist_20261019.csv：全 1,583 位×3 平台；未找到不當作不存在，貼文與追蹤數留空。
- data/processed/social_directory_unmatched_20261019.csv：聯盟候選人頁未唯一對應研究名冊，不併入。
- report/candidate_social_links_20261019.md：可點選的閱讀清單。
資料用途為 collection index，不是策略分析資料。說明與逐檔驗證：data/processed/social_accounts_20261019_documentation.ipynb。保留舊版工作單與主表狀態，不覆寫原始資料。


In [1]:

from pathlib import Path
import json,hashlib,re,unicodedata
from urllib.parse import urlparse,urlunparse,parse_qs,urlencode,urljoin
import pandas as pd
from bs4 import BeautifulSoup
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
PINS={'data/raw/round3_20261019/capture_manifest.json': 'a701143ad22134673e670515332b9691ba9d3f911c9903671e290079509de24f', 'data/processed/candidate_master_reviewed_20261019.csv': '8beb9e6d8c766f9b1484f5c2bc71a1bbf3702640be2135a3abb907d1e6e96177', 'data/processed/candidate_campaign_sources.csv': '682e1fffd9671ae2182548f07a0894b489d8edd1803f4b723106b991115341bd'}
for p,h in PINS.items(): assert sha(ROOT/p)==h, p
manifest=json.loads((ROOT/'data/raw/round3_20261019/capture_manifest.json').read_text())
for r in manifest:
    if r['path']: assert sha(ROOT/r['path'])==r['sha256']
lookup={r['url']:r for r in manifest}
master=pd.read_csv(ROOT/'data/processed/candidate_master_reviewed_20261019.csv',dtype=str).fillna('')
assert len(master)==1583 and master.candidate_key.is_unique
def norm(s): return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺').replace('温','溫')
def match(name,city,mayor=False):
    d=master[master.candidate_name.map(norm).eq(norm(name)) & master.county_city.map(norm).eq(norm(city))]
    d=d[d.office_type.str.contains('長' if mayor else '議員')]
    return d.iloc[0].to_dict() if len(d)==1 else None
def soup(rec): return BeautifulSoup((ROOT/rec['path']).read_bytes(),'html.parser')
def platform(url):
    host=urlparse(url).netloc.lower().removeprefix('www.')
    return {'facebook.com':'Facebook','fb.com':'Facebook','instagram.com':'Instagram','threads.net':'Threads','threads.com':'Threads'}.get(host,'')
def canonical(url):
    u=urlparse(url); host=u.netloc.lower().removeprefix('www.')
    if host=='fb.com':host='facebook.com'
    if host=='threads.net':host='threads.com'
    query=urlencode({'id':parse_qs(u.query)['id'][0]}) if host=='facebook.com' and u.path=='/profile.php' and 'id' in parse_qs(u.query) else ''
    return urlunparse(('https','www.'+host,u.path.rstrip('/'),'',query,''))
rows=[];unmatched=[]
def add(person,url,source,path,status,locator):
    if not person or not platform(url):return
    u=canonical(url)
    parsed=urlparse(u)
    if not parsed.path or parsed.path in ['/share.php','/sharer.php','/sharer','/intent'] or '/posts/' in parsed.path or '/post/' in parsed.path:return
    rows.append(dict(candidate_key=person['candidate_key'],candidate_name=person['candidate_name'],county_city=person['county_city'],office_type=person['office_type'],platform=platform(url),account_url=u,source_url=source,source_file=path,source_locator=locator,evidence_status=status,target_content_status='not_verified_not_posts_captured',checked_date='2026-10-19'))
# Explicit candidate object fields, never page footer or sharing widgets.
js=next(r for r in manifest if 'script_mayor.js' in r['url'])
t=(ROOT/js['path']).read_text(); mayor_data=json.JSONDecoder().raw_decode(t[t.index('['):])[0]
for item in mayor_data:
    person=match(item['name'],item['city'],True)
    if person is None:unmatched.append(dict(name=item['name'],city=item['city'],source=js['url'],reason='no_unique_in_scope_registration'));continue
    for field in ['fb','ig','threads']:
        if item.get(field):add(person,item[field],js['url'],js['path'],'party_candidate_object_link',f"mayor_data name={item['name']} field={field}")
# Alliance candidate cards establish name + city; only candidate-specific social block.
index=lookup['https://www.taiwangogo.tw/candidates/']
for card in soup(index).select('a.cl-cd[href]'):
    name=card.select_one('.nm').get_text(strip=True);city=card['data-city'];person=match(name,city)
    url=urljoin(index['url'],card['href']);rec=lookup.get(url)
    if person is None:
        unmatched.append(dict(name=name,city=city,source=url,reason='no_unique_in_scope_registration'));continue
    if not rec or rec['http_status']!='200':continue
    for a in soup(rec).select('.cv-social a[href]'):
        add(person,a['href'],url,rec['path'],'campaign_directory_candidate_link',f'.cv-social; candidate card {name}; {city}')
# Candidate's own website confirms Puma links independently of party listing.
puma=lookup['https://puma.taipei/']; person=match('沈伯洋','臺北市',True)
for a in soup(puma).select('a[href]'):
    add(person,a['href'],puma['url'],puma['path'],'candidate_website_link','candidate site navigation/footer social links')
# Chiang LINE official account self-links. Government page-wide Facebook links are NOT his account.
chiang=lookup['https://page.line.me/eso5326z'];person=match('蔣萬安','臺北市',True)
assert '蔣萬安' in soup(chiang).get_text()
for a in soup(chiang).select('a[data-testid="facebook"][href],a[data-testid="instagram"][href]'):
    add(person,a['href'],chiang['url'],chiang['path'],'line_official_account_self_link','social media block; LINE account @eso5326z')
# Keep existing, explicitly attributed evidence; do not upgrade verification status.
old=pd.read_csv(ROOT/'data/processed/candidate_campaign_sources.csv',dtype=str).fillna('')
for r in old.to_dict('records'):
    if not platform(r['url']):continue
    d=master[master.candidate_key.eq(r['candidate_key'])]
    if len(d)==1:add(d.iloc[0].to_dict(),r['url'],r['discovered_on'],r['source_file'],'inherited_'+r['verification_status'],'existing candidate_campaign_sources record')
evidence=pd.DataFrame(rows).drop_duplicates(['candidate_key','platform','account_url','source_url']).sort_values(['county_city','candidate_name','platform','account_url'])
assert evidence.candidate_key.isin(master.candidate_key).all()
assert not evidence.account_url.str.contains('moi.gov.tw|humansoftaipei|obasang2018').any()
work=[]
for p in master.to_dict('records'):
    for plat in ['Facebook','Instagram','Threads']:
        d=evidence[evidence.candidate_key.eq(p['candidate_key']) & evidence.platform.eq(plat)]
        urls=list(dict.fromkeys(d.account_url));sources=list(dict.fromkeys(d.source_url))
        work.append(dict(candidate_key=p['candidate_key'],candidate_name=p['candidate_name'],county_city=p['county_city'],office_type=p['office_type'],platform=plat,official_urls=' | '.join(urls),discovered_on=' | '.join(sources),account_status='source_link_found_target_content_pending' if urls else 'not_yet_found_not_absent',posts_status='not_collected_not_zero',window_count='',followers_start='',followers_end='',updated_date='2026-10-19'))
work=pd.DataFrame(work)
assert len(work)==4749 and not work.duplicated(['candidate_key','platform']).any()
out={'evidence':('data/processed/candidate_social_account_evidence_20261019.csv',evidence),'worklist':('data/processed/meta_candidate_account_worklist_20261019.csv',work),'unmatched':('data/processed/social_directory_unmatched_20261019.csv',pd.DataFrame(unmatched))}
verification={}
for key,(path,df) in out.items():
    dest=ROOT/path;assert dest.parent==ROOT/'data/processed' and path not in PINS
    df.to_csv(dest,index=False,encoding='utf-8-sig')
    saved=pd.read_csv(dest,dtype=str).fillna('');assert list(saved.columns)==list(df.columns) and len(saved)==len(df)
    assert len(dest.read_text(encoding='utf-8-sig').splitlines())==len(saved)+1
    verification[path]={'rows':len(saved),'sha256':sha(dest)}
print(json.dumps({'candidates_with_links':int(evidence.candidate_key.nunique()),'candidate_platform_pairs':int(evidence[['candidate_key','platform']].drop_duplicates().shape[0]),'unique_accounts':int(evidence[['candidate_key','platform','account_url']].drop_duplicates().shape[0]),'by_platform':evidence[['candidate_key','platform']].drop_duplicates().platform.value_counts().to_dict(),'outputs':verification},ensure_ascii=False,indent=2))
(ROOT/'data/code/social_directory_20261019_verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2))
def links(d):return '、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.account_url))) or '待補查'
text=['# 候選人社群入口（2026-10-19）','', '先找入口，貼文期間與貼文蒐集暫緩（學生本次指示）。連結有來源依據，但尚未全面檢視帳號內容；不是完整貼文資料庫。空白表示尚未找到，不表示沒有帳號。','',f'本次清單涵蓋 {evidence.candidate_key.nunique()} 位候選人；搜尋來源涵蓋仍不均，不能拿已找到的人當成隨機樣本。','', '## 蔣萬安與沈伯洋','']
def section(key):
    p=master[master.candidate_key.eq(key)].iloc[0];d=evidence[evidence.candidate_key.eq(key)]
    lines=[f"### {p.candidate_name}（{p.county_city}／{p.office_type}）",'', '| Facebook | Instagram | Threads |','| --- | --- | --- |', '| '+' | '.join(links(d[d.platform.eq(plat)]) for plat in ['Facebook','Instagram','Threads'])+' |','']
    lines+=['來源：'+'、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.source_url))), '']
    return lines
for name in ['蔣萬安','沈伯洋']:text+=section(master[master.candidate_name.eq(name)].iloc[0].candidate_key)
text+=['## 其他已找到入口的候選人','', '| 候選人 | 縣市／公職 | Facebook | Instagram | Threads | 核對來源 |','| --- | --- | --- | --- | --- | --- |']
for key,d in evidence.groupby('candidate_key',sort=False):
    p=master[master.candidate_key.eq(key)].iloc[0]
    if p.candidate_name in ['蔣萬安','沈伯洋']:continue
    sources='、'.join(f'[{i+1}]({u})' for i,u in enumerate(dict.fromkeys(d.source_url)))
    text.append('| '+ ' | '.join([p.candidate_name,p.county_city+'／'+p.office_type]+[links(d[d.platform.eq(plat)]) for plat in ['Facebook','Instagram','Threads']]+[sources])+' |')
text+=['', '## 查核界線','', '- 民進黨：以候選人資料物件中的 fb／ig／threads 欄位為依據。','- 台灣前進：以個別候選人頁的 .cv-social 區塊為依據，未把政黨共用頁尾當作個人帳號。','- 蔣萬安：LINE 官方帳號自列 Facebook、Instagram；Threads 仍待找到可靠連結。','- 沈伯洋：競選官網與政黨頁皆有 Threads 連結。Facebook 的數字網址、名稱網址先保留，尚未確認是否為同一頁面。','- 舊來源保留原查核狀態；未宣稱目前可讀取完整內容或帳號仍活躍。','- 不在本研究名冊的聯盟頁人物另列 unmatched 表，沒有強行以相近姓名併入。','- 內政部、臺北市政府共用頁尾與分享按鈕不列為候選人帳號。']
(ROOT/'report/candidate_social_links_20261019.md').write_text('\n'.join(text)+'\n')


{
  "candidates_with_links": 50,
  "candidate_platform_pairs": 132,
  "unique_accounts": 141,
  "by_platform": {
    "Facebook": 50,
    "Instagram": 44,
    "Threads": 38
  },
  "outputs": {
    "data/processed/candidate_social_account_evidence_20261019.csv": {
      "rows": 162,
      "sha256": "b15e1bf88876de049533f381771920c3c560e18eef5c53e04fb8142cbedc5ff9"
    },
    "data/processed/meta_candidate_account_worklist_20261019.csv": {
      "rows": 4749,
      "sha256": "b3432a639e2179f640f68681dc54eb2889fcf743540aef5b71ec4dd1593a0a49"
    },
    "data/processed/social_directory_unmatched_20261019.csv": {
      "rows": 6,
      "sha256": "322af147e7847b92ad7cf9a49929adcae5d86f197cfe69c5be8a655a1b15b786"
    }
  }
}


11566